In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import glob
import os
import fiona
import rasterio
from rasterio.merge import merge
from rasterio.warp import reproject
from rasterio.enums import Resampling
import numpy as np
import geopandas as gpd
from scipy.ndimage import sobel
import joblib 




src_north = rasterio.open("bng_north.tif")
src_south = rasterio.open("bng_south.tif")

# FIX #3: Save nodata value before closing the source files
nodata_val = src_north.nodata

elevation_matrix, elevation_transform = merge([src_north, src_south])

target_crs = src_north.crs.to_string()
target_height = elevation_matrix.shape[1]
target_width = elevation_matrix.shape[2]

src_north.close()
src_south.close()


xmin = elevation_transform[2]
x_pixel_size = elevation_transform[0]
xmax = xmin + (target_width * x_pixel_size)
ymax = elevation_transform[5]
y_pixel_size = elevation_transform[4]
ymin = ymax + (target_height * y_pixel_size)

elevation_grid = elevation_matrix[0].astype(np.float32)
if nodata_val is not None:
    elevation_grid[elevation_grid == nodata_val] = np.nan

slope_x = sobel(elevation_grid, axis=1)
slope_y = sobel(elevation_grid, axis=0)
slope = np.hypot(slope_x, slope_y)

inputdir = "chirps_precipitation_2025"
filepaths = sorted(glob.glob(os.path.join(inputdir, "chirps-v2.0.*.cog")))

max_rain = np.zeros((target_height, target_width), dtype=np.float32)
total_rain = np.zeros((target_height, target_width), dtype=np.float32)

for path in filepaths:
    with rasterio.open(path) as src_rain:
        day_grid = np.empty((target_height, target_width), dtype=np.float32)

        # Resample the 5.5km global file down to your 30m grid
        reproject(
            source=rasterio.band(src_rain, 1),
            destination=day_grid,
            src_transform=src_rain.transform,
            src_crs=src_rain.crs,
            dst_transform=elevation_transform,
            dst_crs=target_crs,
            resampling=Resampling.bilinear
        )

        # Update running summaries pixel-by-pixel
        max_rain = np.maximum(max_rain, day_grid)
        total_rain += day_grid


mean_rain = total_rain / len(filepaths) if filepaths else np.zeros_like(total_rain)

X_full_grid = pd.DataFrame({
    'elevation': elevation_grid.flatten(),
    'max_rainfall': max_rain.flatten(),
    'mean_rainfall': mean_rain.flatten(),
    'total_rainfall': total_rain.flatten(),
    'slope': slope.flatten()
})

fiona.drvsupport.supported_drivers['KML'] = 'rw'   # enable KML driver

gdf_floods_bbmp = gpd.read_file("flood_points.kml", driver='KML')

gdf_floods_bbmp = gdf_floods_bbmp.to_crs(target_crs)

y_full_grid = np.zeros(elevation_grid.size, dtype=np.uint8)
flood_indices = []

for geom in gdf_floods_bbmp.geometry:
    if geom.geom_type == "Point":
        lon, lat = geom.x, geom.y

        # FIX #1: ~transform * (x, y) returns (col, row), not (row, col)
        col, row = ~elevation_transform * (lon, lat)
        row, col = int(np.round(row)), int(np.round(col))

        if 0 <= row < target_height and 0 <= col < target_width:
            flat_index = (row * target_width) + col
            flood_indices.append(flat_index)


if not flood_indices:
    raise ValueError("No valid flood points found in the KML file. Check your data.")

y_full_grid[flood_indices] = 1

flood_indices = np.where(y_full_grid == 1)[0]
non_flood_indices = np.where(y_full_grid == 0)[0]

np.random.seed(50)

sampled_non_flood_indices = np.random.choice(
    non_flood_indices,
    size=len(flood_indices) * 2,
    replace=False
)

final_indices = np.concatenate([flood_indices, sampled_non_flood_indices])

X_balanced = X_full_grid.iloc[final_indices]
y_balanced = y_full_grid[final_indices]

X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced,
    test_size=0.2,
    random_state=50,
    stratify=y_balanced
)

rf_model = RandomForestClassifier(n_estimators=200, max_depth = 10 , min_samples_leaf=5, 
                                   class_weight='balanced',random_state=50, n_jobs=-1)
rf_model.fit(X_train, y_train)
joblib.dump(rf_model, 'flood_model.pkl')
print("Model saved!")

y_pred = rf_model.predict(X_test)

print("\n================ MODEL PERFORMANCE ================")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Metrics Breakdown:")
print(classification_report(y_test, y_pred))


Model saved!

================ MODEL PERFORMANCE ================
Accuracy Score: 90.91%

Classification Metrics Breakdown:
              precision    recall  f1-score   support

           0       0.96      0.90      0.93        51
           1       0.83      0.92      0.87        26

    accuracy                           0.91        77
   macro avg       0.89      0.91      0.90        77
weighted avg       0.91      0.91      0.91        77



In [ ]:
import requests
import matplotlib as plt

def get_bengaluru_forecast():
    url =  "https://api.open-meteo.com/v1/forecast"
    params = { 
        "latitude": 12.9716,
        "longitude": 77.5946,
        "daily": "precipitation_sum",
        "timezone": "Asia/Kolkata",
        "forecast_days": 3
        }
    
    response = requests.get(url, params=params)
    data = response.json()

    daily_rain = data['daily']['precipitation_sum']
    dates = data['daily']['time']

    total = sum(daily_rain)
    mean = np.mean(daily_rain)
    maximum = max(daily_rain)


    print(f"\n---------Live forecast of three days------------------")
    for date, rain in zip(dates, daily_rain):
        print(f"{date}: {rain:.1f}mm")

    print(f"Total: {total:.1f}mm | Mean: {mean:.1f}mm | Peak: {maximum:.1f}mm")

    return total, mean, maximum

total, mean, maximum = get_bengaluru_forecast()


forecast_grid = pd.DataFrame({
    'elevation':      X_full_grid['elevation'],
    'max_rainfall':   maximum,
    'mean_rainfall':  mean,
    'total_rainfall': total,
    'slope':          X_full_grid['slope']
})

flood_probability = rf_model.predict_proba(forecast_grid)[:, 1]
flood_risk_map = flood_probability.reshape(target_height, target_width)

plt.figure(figsize=(12, 10))
plt.imshow(flood_risk_map, cmap='RdYlGn_r')
plt.colorbar(label='Flood Probability')
plt.title(f'Bengaluru Flood Risk Map\nForecast Rainfall: {total:.1f}mm over 3 days')
plt.axis('off')
plt.tight_layout()
plt.savefig('flood_risk_forecast.png', dpi=150)
plt.show()

high_risk   = (flood_probability > 0.7).sum()
medium_risk = ((flood_probability > 0.4) & (flood_probability <= 0.7)).sum()
low_risk    = (flood_probability <= 0.4).sum()

print(f"\n===== ROAD CLOGGING RISK SUMMARY =====")
print(f"High Risk   (>70%): {high_risk:,} pixels  ({high_risk * 900 / 1e6:.2f} sq km)")
print(f"Medium Risk (40-70%): {medium_risk:,} pixels ({medium_risk * 900 / 1e6:.2f} sq km)")
print(f"Low Risk    (<40%): {low_risk:,} pixels  ({low_risk * 900 / 1e6:.2f} sq km)")





---------Live forecast of three days------------------
2026-05-21: 2.7mm
2026-05-22: 7.1mm
2026-05-23: 4.2mm
Total: 14.0mm | Mean: 4.7mm | Peak: 7.1mm
